In [ ]:
# !pip install tensorflow
# !pip install livelossplot

1. 1 - 2 done
2. 3 - 4 done
3. 5 - 6 done
4. 7 - 8 done
5. 9 - 10 done

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, recall_score
from sklearn.utils import shuffle
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import StratifiedKFold
from livelossplot import PlotLossesKerasTF

In [ ]:
def load_datasets(start = 2, end = 10):
    dataset = np.load("/content/drive/MyDrive/Deep Learning 1 Project/Dataset/UrbanSound8K_MelSpectograms 1.npz")

    X = dataset["X"]

    y = dataset["y"]

    for i in range(start, end + 1):

        dataset = np.load(f"/content/drive/MyDrive/Deep Learning 1 Project/Dataset/UrbanSound8K_MelSpectograms {i}.npz")

        X = np.concatenate((X, dataset["X"]), axis = 0)

        y = np.concatenate((y,dataset["y"]), axis = 0)

    return (X , y)

In [ ]:
version = 5
start = 1
end = 2

X, y = load_datasets(start = start, end = end)

print(X.shape)
print(y.shape)

(31658, 128, 44)
(31658,)


In [ ]:
classes = np.unique(y)

print(len(classes))

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y
)

class_weights = dict(zip(classes, weights))

10


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

In [ ]:
model = load_model(f"/content/drive/MyDrive/Deep Learning 1 Project/Models/sound_detect_1.5M_v{version - 1}.keras")

In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=30,
    batch_size=32,
    class_weight = class_weights,
    callbacks = [EarlyStopping(monitor = "val_accuracy", patience=5, restore_best_weights=True), PlotLossesKerasTF()]
)

In [ ]:
model.evaluate(X_test, y_test)

In [ ]:
y_pred = np.argmax(model.predict(X_test), axis=1)

print(classification_report(y_test, y_pred))

In [ ]:
model.save(f'sound_detect_1.5M_v{version}.keras')

In [ ]:
model = models.Sequential()

model.add(layers.Input(shape=(128, 44, 1)))

model.add(layers.Conv2D(32, (3,3), padding='same', activation='relu'))
model.add(layers.BatchNormalization())
model.add(layers.MaxPooling2D((2,2)))

model.add(layers.Conv2D(64, (3,3), padding='same', activation='relu'))
model.add(layers.BatchNormalization())
model.add(layers.MaxPooling2D((2,2)))

model.add(layers.Conv2D(128, (3,3), padding='same', activation='relu'))
model.add(layers.BatchNormalization())
model.add(layers.MaxPooling2D((2,2)))

model.add(layers.Flatten())

model.add(layers.Dense(128, activation='relu'))
model.add(layers.Dropout(0.3))

model.add(layers.Dense(10, activation='softmax'))

model.summary()

In [ ]:
optimizer = optimizers.Adam()
model.compile(
    optimizer=optimizer,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=30,
    batch_size=32,
    class_weight = class_weights,
    callbacks = [EarlyStopping(monitor = "val_accuracy", patience=5, restore_best_weights=True), PlotLossesKerasTF()]
)

In [ ]:
model.evaluate(X_test, y_test)

In [ ]:
y_pred = np.argmax(model.predict(X_test), axis=1)

print(classification_report(y_test, y_pred))

In [ ]:
model = load_model("/content/sound_detect_1.5M_v5.keras")

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
def build_model():
  model = models.Sequential()

  model.add(layers.Input(shape=(128, 44, 1)))

  model.add(layers.Conv2D(32, (3,3), padding='same', activation='relu'))
  model.add(layers.BatchNormalization())
  model.add(layers.MaxPooling2D((2,2)))

  model.add(layers.Conv2D(64, (3,3), padding='same', activation='relu'))
  model.add(layers.BatchNormalization())
  model.add(layers.MaxPooling2D((2,2)))

  model.add(layers.Conv2D(128, (3,3), padding='same', activation='relu'))
  model.add(layers.BatchNormalization())
  model.add(layers.MaxPooling2D((2,2)))

  model.add(layers.Flatten())

  model.add(layers.Dense(128, activation='relu'))
  model.add(layers.Dropout(0.3))

  model.add(layers.Dense(10, activation='softmax'))

  optimizer = optimizers.Adam()
  model.compile(
      optimizer=optimizer,
      loss="sparse_categorical_crossentropy",
      metrics=["accuracy"]
  )

  return model


In [ ]:

fold_accuracies = []

for train_idx, val_idx in skf.split(X, y):

    print("=" * 70 + "\n")
    print("fold number: ", len(fold_accuracies) + 1)

    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    model = build_model()

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )

    model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=30,
        batch_size=32,
        class_weight=class_weights,
        callbacks=[early_stop],
        verbose=1
    )

    val_loss, val_accuracy = model.evaluate(X_val, y_val)
    fold_accuracies.append(val_accuracy)

    print("Validation Loss: ", val_loss)
    print("Validation Accuracy: ", val_accuracy)



fold number:  1
Epoch 1/30
792/792 ━━━━━━━━━━━━━━━━━━━━ 23s 18ms/step - accuracy: 0.6423 - loss: 1.1233 - val_accuracy: 0.6396 - val_loss: 1.0957
Epoch 2/30
792/792 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - accuracy: 0.8137 - loss: 0.5729 - val_accuracy: 0.8307 - val_loss: 0.5287
Epoch 3/30
792/792 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - accuracy: 0.8635 - loss: 0.4076 - val_accuracy: 0.9507 - val_loss: 0.1784
Epoch 4/30
792/792 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.8753 - loss: 0.4122 - val_accuracy: 0.9477 - val_loss: 0.2120
Epoch 5/30
792/792 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - accuracy: 0.9125 - loss: 0.3076 - val_accuracy: 0.9697 - val_loss: 0.1156
Epoch 6/30
792/792 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - accuracy: 0.9285 - loss: 0.2120 - val_accuracy: 0.9319 - val_loss: 0.2120
Epoch 7/30
792/792 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.9313 - loss: 0.2417 - val_accuracy: 0.8762 - val_loss: 0.5133
Epoch 8/30
792/792 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - accuracy: 0.9378 - loss:

In [ ]:
fold_accuracies = np.array(fold_accuracies)

print("Average accuracy: ", fold_accuracies.mean())

Average accuracy:  0.9929559588432312
Average crash recall, class 0:  nan


/tmp/ipykernel_2359/1700205904.py:5: RuntimeWarning: Mean of empty slice.
  print("Average crash recall, class 0: ", crash_recalls.mean())
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [ ]:
model.save('sound_detect_1.5M_v1.keras')